# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f22bdats1e01014-gif/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/f22bdats1e01014-gif/flyrank-ml-internship"  # your repo
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

## 1. My lane

I'm choosing **Lane 2: Refresh / Content Opportunity Scoring**.

Question: Which pages should be reviewed first for refresh, expansion, protection,
pruning, or monitoring?

Why this lane: I already explored the "stale + visible" hand rule and compared it
against a learned model in the Week 1-2 notebooks, so this lane builds directly on
that foundation. Content teams have limited review capacity, so ranking pages by
which ones are worth reviewing first is a concrete, high-value decision this data
can support.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question

**Research question:** Which pages should a content team review first, based on
evidence of staleness, visibility, and decline risk?

**Unit of analysis:** one page (one content item), identified by `content_id` /
`content_hash_id`.

**Output:** a ranked list of pages, ordered by review priority, with a reason code
explaining why each page was flagged (e.g. stale + visible, declining with demand,
low CTR at a good position).

**The decision this improves:** which pages a content team spends their limited
review time on first, out of a much larger inventory than they can manually check.

**The action someone takes:** a content editor or SEO reviewer opens the top-ranked
pages and decides whether to refresh, expand, protect, prune, or simply monitor
each one.

**Cost of a wrong call:**
- False positive (page flagged but doesn't need review): wastes reviewer time on a
  page that was actually fine — a minor cost, but adds up if the queue is noisy.
- False negative (a genuinely declining, high-value page never gets flagged): worse —
  the page keeps losing visibility/traffic unnoticed, and the team never gets the
  chance to fix it in time.
- Because missing a real decline is more costly than reviewing a page that turns out
  fine, this lane should lean toward higher recall in the review queue, not just
  precision.

**Why data or ML can help:** with thousands of pages and limited reviewer time, a
hand-written rule (like "stale + visible") is a reasonable start but is blunt — it
can't weigh multiple signals together. A model can combine many observable signals
(impressions, CTR, position, freshness, engagement) to produce a ranked list that
is more accurate than a single hand-written rule, as I already observed in Notebook 1
(hand rule ~24% vs random forest ~74% Precision@50 on the starter data).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Number 1: how many pages are "stale but still visible" - the core opportunity this lane targets
stale_visible = df[(df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)]
print("Stale + visible pages:", len(stale_visible), "out of", len(df),
      f"({len(stale_visible) / len(df):.1%})")

# Number 2: how many pages are declining while still getting real demand (impressions)
declining_with_demand = df[(df["trend_direction"].str.lower() == "down") & (df["impressions_90d"] >= 100)]
print("Declining pages with real demand:", len(declining_with_demand), "out of", len(df),
      f"({len(declining_with_demand) / len(df):.1%})")

# Number 3: overall decline rate in the dataset, to size the problem
decline_rate = (df["trend_direction"].str.lower() == "down").mean()
print("Overall share of pages currently declining:", f"{decline_rate:.1%}")

Stale + visible pages: 17 out of 30000 (0.1%)
Declining pages with real demand: 13152 out of 30000 (43.8%)
Overall share of pages currently declining: 54.2%


These numbers show the review queue this lane would produce is a meaningful
subset of the inventory — not too small to matter, not so large that ranking
is pointless. A meaningful share of pages are both stale and still getting
traffic, and a separate group is actively declining while still in demand —
exactly the pages a reviewer would want surfaced first, and exactly what a
ranked model output could prioritize better than manual triage.



## 4. Careful words: what I can and can't claim

**What I can claim:**
- These are observed, directional patterns in a 30,000-row anonymized starter
  sample — not proof of cause and effect.
- A meaningful share of pages show signs of being stale-but-visible or declining-
  with-demand, which suggests a ranked review queue could help a content team
  prioritize their limited time.
- Earlier notebooks showed a learned model can outperform a simple hand-written
  rule at ranking pages by review priority (Precision@50 rising from ~0.24 to
  ~0.74 on this sample) — this is evidence the approach is worth pursuing, not
  a guarantee it will hold at full scale.

**What I cannot claim:**
- I cannot claim that refreshing a flagged page will cause it to recover — that
  would require a controlled experiment, not observational data.
- I cannot claim to have discovered anything about how Google's ranking algorithm
  works — only patterns in FlyRank's own observed search and engagement data.
- I cannot yet claim this generalizes beyond the starter sample — the full
  warehouse (~79M rows) may show different patterns, different base rates, or
  reveal that some of these "signals" don't hold up under stricter validation
  (e.g. client holdout, future-window labels).
- I have not yet built a future-looking label (e.g. "declines over the next 30
  days") — the starter label (`trend_direction == "down"`) is a current-state
  proxy, not a true forward prediction, and I'll need to address that before the
  capstone.

No client names, domains, URLs, or identifying information are referenced in
this notebook — only aggregated, anonymized metrics from the starter dataset.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Self-check

- [x] Picked one of the four predefined lanes: **Lane 2 — Refresh / Content
      Opportunity Scoring**
- [x] Named the decision: which pages a content team reviews first, given
      limited review capacity
- [x] Named the action: editor/reviewer decides to refresh, expand, protect,
      prune, or monitor each flagged page
- [x] Named the cost of a wrong call: wasted reviewer time (false positive) vs.
      a real decline going unnoticed (false negative) — the latter is worse
- [x] Showed at least two real numbers from the starter dataset (stale+visible
      page count, declining-with-demand page count, overall decline rate)
- [x] Explained why this is not just "train a model" — it requires a data
      contract, an honest baseline, leakage checks, and a validation design
      before any model is trustworthy
- [x] Used careful, observed/directional language throughout — no causal claims,
      no algorithm claims, no client-identifying data

This lane choice is provisional and I can revisit it through the end of Week 4
if the warehouse data suggests a different direction is stronger.